# 4. LLM-режим для многоканального случая

**Цель:** запустить детерминированный `GraphEVTPipeline` на Kuramoto-ряде и попросить LLM проверить результат — обнаружение, порог и предполагаемый источник — без синтетической разметки. Kuramoto CSV не содержит колонки события; baseline фиксирован заранее (первые 300 точек), а не выбран по результату.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt. Ключ не записывается в ноутбук или артефакты.

In [1]:
from pathlib import Path
import os
from getpass import getpass
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralAPIError, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_synchronized_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_synchronized_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
assert np.isfinite(x).all() and x.ndim == 2

In [2]:
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
MISTRAL_READY = False
mistral_client = None

if api_key:
    try:
        mistral_client = MistralClient(
            api_key=api_key,
            model="ministral-3b-2512",
            max_retries=0,
        )
        check_reply = mistral_client.complete(
            "Reply exactly with MISTRAL_OK.",
            "Connection check. Reply exactly with MISTRAL_OK.",
        )
        MISTRAL_READY = bool(check_reply.strip())
        print("Mistral API check passed." if MISTRAL_READY else "Mistral returned an empty response.")
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit (HTTP 429). Check your account usage and rate limits, wait, then rerun this cell.")
        else:
            print(f"Mistral API check failed: {error}")
    except Exception as error:
        print(f"Mistral API check failed ({type(error).__name__}).")
else:
    print("Mistral API check skipped: no key was provided.")

Mistral API check failed: Mistral API rejected the credentials (HTTP 401). Check that MISTRAL_API_KEY contains an active API key (not its name, a workspace ID, or the .env.example placeholder), then create a new key in the Mistral console if necessary.


In [3]:
# Kuramoto CSV has no labeled event; the baseline is fixed before looking at the result.
BASELINE_SIZE = 300
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.90, alarm_probability=0.99, top_k=3, persistence=2),
    graph=GraphConfig(method="ar", edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
)
numeric_result = pipeline.run_detailed(x)
detection = numeric_result.detection
plan = numeric_result.plan
estimated_source = None if numeric_result.ranking is None else channel_names[numeric_result.ranking.source]

pd.DataFrame([{
    "baseline_size": BASELINE_SIZE,
    "detected": detection.detected,
    "alarm_index": detection.time_index,
    "alarm_threshold": detection.alarm_threshold,
    "estimated_source": estimated_source,
    "planner_route": None if plan is None else plan.route,
    "completed_actions": None if plan is None else " -> ".join(plan.completed_actions),
    "stop_reason": None if plan is None else plan.stop_reason,
}])

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

,baseline_size,detected,alarm_index,alarm_threshold,estimated_source,planner_route,completed_actions,stop_reason
0,300,True,300,5.49498,channel_08,n-d,inspect_input -> check_stationarity -> detect_...,None


In [4]:
if MISTRAL_READY:
    team = EVTAgentTeam(pipeline, mistral_client)
    try:
        agent_report = team.run(
            x,
            task=(
                "Проверь результат EVT-детекции на многоканальном ряде без разметки: "
                f"detected={detection.detected}, alarm_index={detection.time_index}, "
                f"estimated_source={estimated_source}. "
                "Явно укажи, что источник — оценка ranking, а не подтверждённая причина; "
                "не утверждай отсутствие события только из-за отсутствия тревоги."
            ),
        )
        print(agent_report.final_report)
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit during review. Check account usage/rate limits, wait, then rerun this cell.")
        else:
            print(f"LLM review failed: {error}")
    except Exception as error:
        print(f"LLM review failed ({type(error).__name__}).")
else:
    print("LLM review skipped; the numeric pipeline result is still available.")

LLM review skipped; the numeric pipeline result is still available.


`EVTAgentTeam` вызывает тот же детерминированный pipeline и получает `run_detailed`-результат вместе с plan/act/observe trace. Для n-D входа граф и ranking уже присутствуют, поэтому LLM должен явно отделять их от подтверждённой причины. Генератор задаёт номинальный ведущий канал (`channel_00`) отдельно от detector'а; сравнение с ним — не более чем неформальная проверка, не источник метрики.